# 02 — OMI transactions: from annual releases to municipal NTN

**Goal.** Build one clean annual municipality-level panel of residential **NTN (Numero di Transazioni Normalizzate)** for 2011–2025.

The notebook separates four operations that are easy to confuse: **discover releases → read the residential table → attach municipality names/geography → validate the reported total against the size breakdown**.

> **Why this matters:** the raw files use a different municipality identifier system from the OMI quotation files. Here we preserve the original cadastral-style code; the canonical ISTAT crosswalk belongs to the later integration stage.

## 1. Setup and release discovery

The files are annual. We identify them from the explicit filename pattern rather than slicing arbitrary character positions from filenames. This prevents a small naming change from silently assigning the wrong year.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',50)
pd.set_option('display.float_format',lambda x:f'{x:,.2f}')

def find_project_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'data'/'raw'/'transactions').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/transactions.')

PROJECT_ROOT=find_project_root()
RAW_DIR=PROJECT_ROOT/'data'/'raw'/'transactions'
FILE_RE=re.compile(r'^(?P<year>\d{4})_(?P<table>LISTA-COM|VALORI-RES|VALORI-COM|VALORI-PER)\.csv$',re.I)

files=[]
for path in RAW_DIR.rglob('*.csv'):
    match=FILE_RE.match(path.name)
    if match:
        files.append({'path':path,'year':int(match.group('year')),'table':match.group('table').upper()})
catalogue=pd.DataFrame(files).sort_values(['year','table']).reset_index(drop=True)

expected={'LISTA-COM','VALORI-RES','VALORI-COM','VALORI-PER'}
coverage=catalogue.groupby('year')['table'].agg(lambda s:sorted(set(s))).reset_index()
coverage['missing']=coverage['table'].map(lambda x:sorted(expected-set(x)))

print(f'Releases: {coverage.year.min()}–{coverage.year.max()}')
display(coverage)

## 2. Read the residential NTN table

`VALORI-RES` contains the residential NTN by municipality and size class. Column names vary slightly across years, so the code identifies them by meaning rather than hard-coding one year's exact spelling.

`LISTA-COM` is used as the municipality dictionary: it supplies the municipality name and geography associated with the annual transaction code.

In [ ]:
def read_omi_csv(path):
    return pd.read_csv(path,sep=';',dtype='string',na_values=['','NA','N/A','nan','-'],keep_default_na=True)

def find_column(columns,*patterns):
    for pattern in patterns:
        matches=[c for c in columns if re.search(pattern,str(c),flags=re.I)]
        if matches:
            return matches[0]
    raise KeyError(f'No column matched {patterns}. Available columns: {list(columns)}')

size_patterns={
    'ntn_upto_50':[r'fino\s*a\s*50'],
    'ntn_50_85':[r'50.*85'],
    'ntn_85_115':[r'85.*115'],
    'ntn_115_145':[r'115.*145'],
    'ntn_over_145':[r'oltre\s*145'],
}

def load_year(year):
    res_path=catalogue.loc[(catalogue.year.eq(year)) & catalogue.table.eq('VALORI-RES'),'path'].iloc[0]
    list_path=catalogue.loc[(catalogue.year.eq(year)) & catalogue.table.eq('LISTA-COM'),'path'].iloc[0]
    res=read_omi_csv(res_path)
    municipalities=read_omi_csv(list_path)

    cod_res=find_column(res.columns,rf'^{year}_CodCom$',r'^CodCom$',r'cod[_ ]?com')
    cod_list=find_column(municipalities.columns,rf'^{year}_CodCom$',r'^CodCom$',r'cod[_ ]?com')
    total=find_column(res.columns,rf'^NTN[_ ]*{year}$',rf'^NTN.*{year}$')

    out=res.rename(columns={cod_res:'municipality_code',total:'ntn_res'}).copy()
    for key,patterns in size_patterns.items():
        col=find_column(res.columns,*patterns)
        out[key]=res[col]

    geo=municipalities.rename(columns={cod_list:'municipality_code'}).copy()
    geo_cols=['municipality_code']+[c for c in ['Comune','Regione','Provincia'] if c in geo.columns]
    geo=geo[geo_cols].drop_duplicates('municipality_code')

    out=out.merge(geo,on='municipality_code',how='left',validate='many_to_one')
    out['year']=year

    # Keep only real cadastral-style municipality codes. This removes non-municipality labels such as 2017 NON NATI/NON NATO.
    out=out[out['municipality_code'].astype('string').str.fullmatch(r'[A-Z]\d{3}',na=False)].copy()

    numeric_cols=['ntn_res',*size_patterns.keys()]
    for col in numeric_cols:
        out[col]=pd.to_numeric(out[col].astype('string').str.replace(',','.',regex=False),errors='coerce')

    out['ntn_res_size_sum']=out[list(size_patterns)].sum(axis=1,min_count=1)
    out['ntn_res_gap']=out['ntn_res']-out['ntn_res_size_sum']
    out['ntn_res_source_type']=np.where(out['ntn_res'].notna(),'reported','derived')
    out.loc[out['ntn_res'].isna(),'ntn_res']=out.loc[out['ntn_res'].isna(),'ntn_res_size_sum']
    out['ntn_res_source_type']=np.where(out['ntn_res_source_type'].eq('derived'),'derived','reported')
    return out

years=sorted(catalogue.year.unique())
res_panels=[load_year(year) for year in years]
transactions=pd.concat(res_panels,ignore_index=True)

print(f'Rows: {len(transactions):,}')
print(f'Years: {transactions.year.min()}–{transactions.year.max()}')
print(f'Municipalities/codes: {transactions.municipality_code.nunique():,}')
display(transactions.head())

## 3. Why the reconciliation check is essential

Every residential record contains a reported total NTN and, where available, NTN split across five dwelling-size bands. The two representations should agree.

The check below does **not** overwrite the reported value when it exists. It measures the difference first; only a missing reported total can be filled from the size breakdown.

In [ ]:
size_cols=['ntn_upto_50','ntn_50_85','ntn_85_115','ntn_115_145','ntn_over_145']

max_gap=transactions['ntn_res_gap'].abs().max()
non_zero_gap=(transactions['ntn_res_gap'].abs()>1e-9).sum()
derived_rows=transactions['ntn_res_source_type'].eq('derived').sum()

print(f'Max absolute reconciliation gap: {max_gap:,.6f}')
print(f'Rows with non-zero reconciliation gap: {non_zero_gap:,}')
print(f'Derived total rows: {derived_rows:,}')

if non_zero_gap:
    display(transactions.loc[transactions['ntn_res_gap'].abs()>1e-9,['year','municipality_code','ntn_res','ntn_res_size_sum','ntn_res_gap']].head(20))

annual_control=(transactions.groupby('year',as_index=False).agg(municipalities=('municipality_code','nunique'),ntn_res=('ntn_res','sum'),ntn_res_size_sum=('ntn_res_size_sum','sum'),reported_rows=('ntn_res_source_type',lambda s:s.eq('reported').sum()),derived_rows=('ntn_res_source_type',lambda s:s.eq('derived').sum())))
annual_control['gap']=annual_control['ntn_res']-annual_control['ntn_res_size_sum']
display(annual_control)

## 4. National market dynamics

NTN measures normalized transaction volume. It is therefore a **market activity indicator**, not a price measure. The annual series is the natural frequency for this source: unlike OMI quotations, transactions are annual in this project.

In [ ]:
national=(transactions.groupby('year',as_index=False).agg(ntn_res=('ntn_res','sum'),municipalities=('municipality_code','nunique')).sort_values('year'))
national['ntn_yoy_pct']=national['ntn_res'].pct_change().mul(100)
display(national)

fig,ax=plt.subplots(figsize=(11,5))
ax.plot(national['year'],national['ntn_res'],marker='o')
ax.set(title='Italian residential NTN — annual normalized transactions',xlabel='Year',ylabel='NTN')
ax.grid(alpha=.25)
plt.show()

## 5. Residential size composition

The five size bands allow us to understand *how* transaction volume is distributed, not only how much volume exists. Shares are calculated within each year, so they sum to approximately 100%.

In [ ]:
size_long=(transactions.groupby('year',as_index=False)[size_cols].sum().melt(id_vars='year',var_name='size_band',value_name='ntn'))
size_long['share_pct']=size_long['ntn'].div(size_long.groupby('year')['ntn'].transform('sum')).mul(100)
latest_year=int(size_long.year.max())
display(size_long.loc[size_long.year.eq(latest_year)].sort_values('share_pct',ascending=False).style.format({'ntn':'{:,.2f}','share_pct':'{:,.1f}%'}))

fig,ax=plt.subplots(figsize=(10,5))
latest=size_long.loc[size_long.year.eq(latest_year)].sort_values('share_pct',ascending=False)
ax.bar(latest['size_band'],latest['share_pct'])
ax.set(title=f'Residential NTN composition — {latest_year}',xlabel='Dwelling size band',ylabel='Share of NTN (%)')
ax.tick_params(axis='x',rotation=25)
plt.show()

## 6. What this notebook establishes

- **One row = one municipality-year residential observation.**
- The original annual NTN is retained as the main measure.
- Size-band totals are used as a reconciliation and composition check.
- Non-municipality labels are excluded explicitly.
- The cadastral-style municipality code is preserved as a source identifier.

**Important integration note:** these codes are not the same key used by the OMI quotation and population datasets. The canonical ISTAT crosswalk is intentionally left to the later integration notebook; forcing a numeric transformation here would create false municipality matches.